# Model Context Protocol (MCP) — A Complete Concept Guide

> **School of Core AI** · Managed by Vivek · Reviewed by Ashutosh

> A practical, code-grounded walkthrough of the Model Context Protocol.

---

# 1. What is MCP?

## The N×M Integration Problem

Every AI app needs to talk to external systems — databases, APIs, file stores, search engines. Without a standard, each app writes custom glue code for each system.

```mermaid
graph LR
    subgraph "Without MCP — N apps × M integrations"
        A1["AI App 1"] --> I1["Custom DB code"]
        A1 --> I2["Custom API code"]
        A2["AI App 2"] --> I3["Custom DB code (copy)"]
        A2 --> I4["Custom API code (copy)"]
        A3["AI App 3"] --> I5["Custom DB code (copy again)"]
        A3 --> I6["Custom API code (copy again)"]
    end
```

If you have **N AI apps** and **M external systems**, you write **N × M** integrations. Add a new app? Rewrite all M. Add a new system? Update all N apps.

## What MCP Does

MCP is a **client-server protocol** that standardizes how AI apps discover and call external capabilities.

```mermaid
graph LR
    subgraph "With MCP — N apps + 1 server"
        A1["AI App 1"] --> S["MCP Server"]
        A2["AI App 2"] --> S
        A3["AI App 3"] --> S
        S --> DB["Database"]
        S --> API["External API"]
        S --> FS["File Store"]
    end
```

- **N + M** integrations instead of N × M
- Each app discovers capabilities at runtime (`tools/list`, `resources/list`)
- The server owns auth, validation, audit — not each app
- Add a new app = just connect to the server
- Add a new capability = add it once on the server

## The Three Capability Types

| Type | What it is | Example | Analogy |
|------|-----------|---------|---------|
| **Tools** | Functions the AI can call (with side effects) | `create_lead()`, `search_courses()` | POST endpoints |
| **Resources** | Read-only data the AI can read | `scai://catalog/courses` | GET endpoints |
| **Prompts** | Pre-written prompt templates | `admissions_qualify_enquiry` | Stored procedures for LLM context |

> **Key insight:** Tools are for *actions*, resources are for *context*, prompts are for *consistency*.

---

# 2. Why MCP? (vs alternatives)

## MCP vs Direct DB Access

| Concern | Direct DB | MCP |
|--------|-----------|-----|
| DB credentials | In every client | Server only |
| Auth/RBAC | Each app implements | Centralized |
| Audit trail | Each app logs | One audit boundary |
| Input validation | Manual | Pydantic schemas enforced by server |
| Write safety | App decides | Server enforces confirmation gates |
| New client | Rewrite all DB code | Just connect to :8010 |
| Change business logic | Update every app | Update server once |

## MCP vs REST APIs

| Concern | REST | MCP |
|--------|------|-----|
| Discovery | Read OpenAPI docs manually | `tools/list` at runtime |
| Schema | OpenAPI spec (static) | JSON Schema in protocol (dynamic) |
| Auth | Per-endpoint | Protocol-level (JWT in headers) |
| Context for AI | Not designed for it | Built for LLM tool-calling |
| Prompt templates | No | Yes — `prompts/list` |
| Resources | Just endpoints | Typed URI scheme (`scai://...`) |

## MCP vs Function Calling (OpenAI/Anthropic)

| Concern | Function calling | MCP |
|--------|-----------------|-----|
| Where tools live | Hardcoded in app | Discovered at runtime from server |
| Schema | Per-model JSON | Protocol-standard JSON Schema |
| Multi-model | Rewrite per provider | Same server works with any LLM |
| Server-side logic | Client runs everything | Server owns business logic |

> **MCP is not a replacement for function calling — it's a way to *externalize* and *standardize* the tools that function calling invokes.**

---

# 3. How MCP Works

## Protocol Lifecycle

```mermaid
sequenceDiagram
    participant Client as AI App (Client)
    participant Server as MCP Server

    Client->>Server: initialize (protocolVersion, capabilities)
    Server-->>Client: result (serverInfo, capabilities)
    Note over Client,Server: Session established

    Client->>Server: tools/list
    Server-->>Client: [tool1, tool2, ...] with JSON schemas

    Client->>Server: resources/list
    Server-->>Client: [resource URIs]

    Client->>Server: prompts/list
    Server-->>Client: [prompt templates]

    Note over Client: User asks question
    Note over Client: LLM decides which tool to call

    Client->>Server: tools/call (name="search_courses", args={query: "ai"})
    Note over Server: 1. Auth (JWT)  2. RBAC  3. Validate  4. Execute  5. Audit
    Server-->>Client: result (typed output)

    Note over Client: LLM composes answer from verified data
```

## Transports

| Transport | Use case | Bidirectional |
|-----------|---------|--------------|
| **Streamable HTTP** | Independent processes, network | Yes (SSE for server→client) |
| **stdio** | Local CLI, MCP Inspector | Yes (stdin/stdout) |

> The 2026-07-28 spec **removed** the old SSE transport. Streamable HTTP is now the standard for network communication. Each request is stateless — cross-call state is server-minted IDs (`quote_id`, `approval_id`, `lead_id`).

## The Confirmation Gate (Critical Safety Pattern)

MCP servers can enforce a **prepare → confirm** pattern for writes:

```mermaid
graph LR
    U["User: 'Book a callback'"] --> C["Client: calls leads_prepare"]
    C --> S["Server: creates pending approval (no lead yet)"]
    S --> C2["Client: shows preview to user"]
    C2 --> G{"User confirms?"}
    G -->|Yes| S2["Server: leads_confirm_create → creates lead"]
    G -->|No| S3["Server: approval expires (no lead created)"]
```

> **Without this gate, an LLM hallucination could create a lead just because the model said "callback booked." The gate ensures a human confirms every write.**

---

# 4. Why NOT MCP? (When it's overkill)

MCP adds complexity. Use it only when the trade-off is worth it.

| Situation | Use MCP? | Why |
|-----------|----------|-----|
| 1 AI app, 3 local Python functions | ❌ No | Just call the functions directly |
| 1 AI app, 1 database | ❌ No | Direct DB or simple REST is simpler |
| 2+ AI apps sharing the same backend | ✅ Yes | Avoid duplicating integrations |
| Need runtime capability discovery | ✅ Yes | `tools/list` is the whole point |
| Need centralized auth/audit | ✅ Yes | Server owns the security boundary |
| Need a confirmation gate for writes | ✅ Yes | Server enforces it, not each client |
| Different LLM frameworks (LangGraph + raw API) | ✅ Yes | MCP is framework-agnostic |
| Prototyping a quick demo | ❌ No | Ship fast, add MCP when you have 2+ clients |

> **Rule of thumb:** The second independent client is what justifies MCP. One client = overhead. Two clients = savings. Three+ = essential.

## The School of Core AI proof

This project has **two independent clients** (learner host + counsellor host) calling the **same server**. That's the minimum to demonstrate MCP's value. A no-MCP comparison demo (`ui/no_mcp_demo.py`) shows what you'd lose without it.

---

# 6. Use Cases — School of Core AI Project Mapping

## Architecture

```mermaid
graph TB
    subgraph "AI Clients (independent apps)"
        LH["Learner Host<br/>:8020 — LangGraph<br/>learner JWT"]
        CH["Counsellor Host<br/>:8030 — LangGraph<br/>counsellor JWT"]
    end

    subgraph "MCP Server :8010"
        AUTH["JWT Auth"]
        RBAC["RBAC (role → allowed tools)"]
        VAL["Pydantic Validation"]
        IDEM["Idempotency Check"]
        AUDIT["Audit Log"]
        DOMAIN["Domain Services<br/>(catalog, fees, leads, callbacks)"]
        RES["Resources<br/>(catalog, policies, KB, schemas)"]
    end

    subgraph "PostgreSQL :5433"
        DB["courses, batches, fees,<br/>leads, callbacks, audit"]
    end

    LH -->|"HTTP JSON-RPC"| AUTH
    CH -->|"HTTP JSON-RPC"| AUTH
    AUTH --> RBAC --> VAL --> IDEM --> DOMAIN --> DB
    AUDIT --> DB
    RES --> DB
```

## Tool Catalogue (11 tools)

| Tool | What it does | Learner | Counsellor |
|------|-------------|---------|------------|
| `catalog_search_courses` | Search published courses | ✅ | ✅ |
| `catalog_get_course` | Get course details | ✅ | ✅ |
| `batches_find_upcoming` | List upcoming batches | ✅ | ✅ |
| `fees_create_quote` | Generate a fee quote | ✅ | ✅ |
| `policies_get_current` | Read admissions policy | ✅ | ✅ |
| `leads_prepare` | Create pending approval | ✅ | ❌ |
| `leads_confirm_create` | Create lead from approval | ✅ | ❌ |
| `callbacks_schedule` | Schedule a callback | ✅ | ✅ |
| `leads_list_assigned` | List counsellor's leads | ❌ | ✅ |
| `leads_get_summary` | Get lead details | ❌ | ✅ |
| `leads_update_stage` | Update lead stage | ❌ | ✅ |

> **RBAC in action:** The learner can *create* leads (prepare + confirm) but can't *see* other people's leads. The counsellor can *read and update* leads but can't *create* them.

## Resources (8 read-only)

| URI | What it provides |
|-----|-----------------|
| `scai://catalog/courses` | Full course catalogue snapshot |
| `scai://catalog/courses/{id}` | Single course detail |
| `scai://policies/{slug}/current` | Active policy text |
| `scai://schemas/lead-intake` | Lead intake JSON schema |
| `scai://schemas/fee-quote` | Fee quote output schema |
| `scai://kb/articles` | Knowledge Base — all article summaries |
| `scai://kb/articles/{id}` | Knowledge Base — full article body |
| `scai://kb/search?q=...` | Knowledge Base — keyword search |

## The confirmation gate

| Step | Learner Host | MCP Server | DB |
|------|-------------|------------|-----|
| 1. User says "I want a callback" | Calls `leads_prepare` | Creates `LeadApproval` (status=pending) | No lead row yet |
| 2. User sees preview | Shows ✅/❌ buttons | — | — |
| 3. User clicks ✅ | Calls `leads_confirm_create` | Checks approval → creates `Lead` row | Lead row inserted |
| 4. Audit | — | Logs `leads.confirm_create` (actor, result, latency) | `ToolAuditEvent` row |

> **The key:** Step 1 creates NO lead. Only step 3 does. If the user clicks ❌ or closes the browser, the approval expires and nothing was written.

---

# 6. Use Cases — SCAI Admissions Project Mapping

## Architecture

```mermaid
graph TB
    subgraph "AI Clients (independent apps)"
        LH["Learner Host<br/>:8020 — LangGraph<br/>learner JWT"]
        CH["Counsellor Host<br/>:8030 — LangGraph<br/>counsellor JWT"]
    end

    subgraph "MCP Server :8010"
        AUTH["JWT Auth"]
        RBAC["RBAC (role → allowed tools)"]
        VAL["Pydantic Validation"]
        IDEM["Idempotency Check"]
        AUDIT["Audit Log"]
        DOMAIN["Domain Services<br/>(catalog, fees, leads, callbacks)"]
        RES["Resources<br/>(catalog, policies, KB, schemas)"]
    end

    subgraph "PostgreSQL :5433"
        DB["courses, batches, fees,<br/>leads, callbacks, audit"]
    end

    LH -->|"JSON-RPC over HTTP"| AUTH
    CH -->|"JSON-RPC over HTTP"| AUTH
    AUTH --> RBAC --> VAL --> IDEM --> DOMAIN --> DB
    AUDIT --> DB
    RES --> DB
```

## Tool Catalogue (11 tools)

| Tool | What it does | Learner | Counsellor |
|------|-------------|---------|------------|
| `catalog_search_courses` | Search published courses | ✅ | ✅ |
| `catalog_get_course` | Get course details | ✅ | ✅ |
| `batches_find_upcoming` | List upcoming batches | ✅ | ✅ |
| `fees_create_quote` | Generate a fee quote | ✅ | ✅ |
| `policies_get_current` | Read admissions policy | ✅ | ✅ |
| `leads_prepare` | Create pending approval | ✅ | ❌ |
| `leads_confirm_create` | Create lead from approval | ✅ | ❌ |
| `callbacks_schedule` | Schedule a callback | ✅ | ✅ |
| `leads_list_assigned` | List counsellor's leads | ❌ | ✅ |
| `leads_get_summary` | Get lead details | ❌ | ✅ |
| `leads_update_stage` | Update lead stage | ❌ | ✅ |

> **RBAC in action:** The learner can *create* leads (prepare + confirm) but can't *see* other people's leads. The counsellor can *read and update* leads but can't *create* them.

## Resources (8 read-only)

| URI | What it provides |
|-----|-----------------|
| `scai://catalog/courses` | Full course catalogue snapshot |
| `scai://catalog/courses/{id}` | Single course detail |
| `scai://policies/{slug}/current` | Active policy text |
| `scai://schemas/lead-intake` | Lead intake JSON schema |
| `scai://schemas/fee-quote` | Fee quote output schema |
| `scai://kb/articles` | Knowledge Base — all article summaries |
| `scai://kb/articles/{id}` | Knowledge Base — full article body |
| `scai://kb/search?q=...` | Knowledge Base — keyword search |

## The confirmation gate in the SCAI project

| Step | Learner Host | MCP Server | DB |
|------|-------------|------------|-----|
| 1. User says "I want a callback" | Calls `leads_prepare` | Creates `LeadApproval` (status=pending) | No lead row yet |
| 2. User sees preview | Shows ✅/❌ buttons | — | — |
| 3. User clicks ✅ | Calls `leads_confirm_create` | Checks approval → creates `Lead` row | Lead row inserted |
| 4. Audit | — | Logs `leads.confirm_create` (actor, result, latency) | `ToolAuditEvent` row |

> **The key:** Step 1 creates NO lead. Only step 3 does. If the user clicks ❌ or closes the browser, the approval expires and nothing was written.

---

# 7. MCP Flow Diagram (Mermaid → PNG)

The code cell below renders the MCP architecture as a Mermaid diagram and saves it as a PNG file (`mcp_flow_diagram.png`).

In [ ]:
# Generate the MCP architecture flow diagram as a PNG
# Uses the Mermaid CLI (mmdc) via subprocess. If not installed, falls back to
# writing a .mmd file you can render at https://mermaid.live

import subprocess
import sys
from pathlib import Path

MERMAID_DIAGRAM = """
graph TB
    subgraph Clients["AI Clients (independent apps)"]
        LH["🎓 Learner Host<br/>:8020 — LangGraph<br/>JWT: learner role"]
        CH["🎧 Counsellor Host<br/>:8030 — LangGraph<br/>JWT: counsellor role"]
    end

    subgraph Server["MCP Server :8010"]
        AUTH["JWT Auth"]
        RBAC["RBAC<br/>role → allowed tools"]
        VAL["Pydantic Validation"]
        IDEM["Idempotency Check"]
        DOMAIN["Domain Services<br/>catalog · fees · leads · callbacks"]
        RES["Resources<br/>catalog · policies · KB · schemas"]
        AUDIT["Audit Log"]
    end

    subgraph DB["PostgreSQL :5433"]
        TABLES["courses · batches · fees<br/>leads · callbacks · audit"]
    end

    LH -->|"JSON-RPC / HTTP"| AUTH
    CH -->|"JSON-RPC / HTTP"| AUTH
    AUTH --> RBAC
    RBAC --> VAL
    VAL --> IDEM
    IDEM --> DOMAIN
    DOMAIN --> TABLES
    RES --> TABLES
    AUDIT --> TABLES

    style LH fill:#4CAF50,color:#fff
    style CH fill:#2196F3,color:#fff
    style AUTH fill:#FF9800,color:#fff
    style DOMAIN fill:#9C27B0,color:#fff
    style TABLES fill:#607D8B,color:#fff
"""

output_dir = Path.cwd()
mmd_path = output_dir / "mcp_flow_diagram.mmd"
png_path = output_dir / "mcp_flow_diagram.png"

# Write the .mmd file
mmd_path.write_text(MERMAID_DIAGRAM, encoding="utf-8")
print(f"✅ Mermaid source written: {mmd_path}")

# Try to render with mmdc (Mermaid CLI)
try:
    result = subprocess.run(
        ["mmdc", "-i", str(mmd_path), "-o", str(png_path), "-t", "default", "-b", "white"],
        capture_output=True, text=True, timeout=30
    )
    if result.returncode == 0:
        print(f"✅ PNG saved: {png_path}")
    else:
        print(f"⚠️  mmdc failed: {result.stderr.strip()}")
        print(f"   Render manually: npx @mermaid-js/mermaid-cli -i {mmd_path} -o {png_path}")
        print(f"   Or paste the diagram at: https://mermaid.live")
except FileNotFoundError:
    print("⚠️  mmdc not installed. Install with: npm install -g @mermaid-js/mermaid-cli")
    print(f"   Or paste the diagram at: https://mermaid.live")
except Exception as e:
    print(f"⚠️  Error: {e}")
    print(f"   Or paste the diagram at: https://mermaid.live")

---

## References

- [MCP Specification (2026-07-28)](https://modelcontextprotocol.io/specification/2026-07-28)
- [Official MCP Python SDK](https://github.com/modelcontextprotocol/python-sdk)
- [LangGraph Documentation](https://docs.langchain.com/oss/python/langgraph)

---

> **School of Core AI** · Managed by Vivek · Reviewed by Ashutosh
>
> *"The demo succeeds only if the database state and audit trail support every claim made on screen."*